# Name Hierarchy Levels — iGEM Teams (Mid & High)

Assigns globally unique, publication-ready names to the mid-level and high-level
topic groups for **iGEM Teams**. For each group the LLM sees the low-level
sub-topics (name + description) it contains, then returns one name per group via
OpenAI function calling. Prompts come from `prompts_hierarchy.yaml`.

> Run `get_topic_hierarchy.ipynb` (this folder) **first**.

**Updates** `assets/reports/teams_topic_name_hierarchy.tsv` with `mid_name`
and `high_name` columns.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 04-topic_hierarchy/, where the aux/
# package resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from aux.paths import REPORTS_DIR, OPENAI_MODEL, set_seed
from aux.naming import (
    load_prompts, make_client, build_system_prompt,
    name_hierarchy_level, load_naming_inputs, save_named_hierarchy,
)

set_seed()

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
PREFIX = "teams"

prompts = load_prompts()
client = make_client()
system_prompt = build_system_prompt(prompts)

topic_names, hierarchy = load_naming_inputs(PREFIX)
print(f"{PREFIX}: {len(topic_names)} low-level topics | "
      f"mid groups {hierarchy[hierarchy['mid'] >= 0]['mid'].nunique()} | "
      f"high groups {hierarchy[hierarchy['high'] >= 0]['high'].nunique()}")

teams: 161 low-level topics | mid groups 44 | high groups 4


## 1. Name the mid- and high-level groups

In [3]:
mid_names = name_hierarchy_level(
    hierarchy, topic_names, level_col="mid", label="mid",
    client=client, system_prompt=system_prompt, model=OPENAI_MODEL,
)
high_names = name_hierarchy_level(
    hierarchy, topic_names, level_col="high", label="high",
    client=client, system_prompt=system_prompt, model=OPENAI_MODEL,
)

  Naming 44 mid groups via gpt-4.1-nano …
  ✓ 44 mid names assigned
  Naming 4 high groups via gpt-4.1-nano …
  ✓ 4 high names assigned


## 2. Add the name columns and save

In [4]:
hierarchy["mid_name"] = hierarchy["mid"].map(mid_names)
hierarchy["high_name"] = hierarchy["high"].map(high_names)
save_named_hierarchy(hierarchy, PREFIX)

print(f"Saved → {REPORTS_DIR / f'{PREFIX}_topic_name_hierarchy.tsv'}")
hierarchy.head(10)

Saved → /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports/teams_topic_name_hierarchy.tsv


,global_name,low,mid,high,mid_name,high_name
0,Spatial and Stimuli-Responsive Gene Regulation,0,0,0,Advanced Gene Regulation and Synthetic Systems,Advanced Gene Regulation & Synthetic Systems
1,Plastic Biodegradation and Recycling,1,1,1,Biodegradation and Environmental Bioremediation,Environmental Bioremediation & Resource Recycling
2,Environmental Monitoring and Bioremediation,2,2,2,Synthetic Diagnostics and Biosensing Technologies,Medical Diagnostics & Therapeutics
3,Bacterial Cancer Diagnostics and Therapy,3,3,2,Microbial-Based Therapeutics and Cancer Applic...,Medical Diagnostics & Therapeutics
4,Synthetic Diagnostics and Forensics,4,2,2,Synthetic Diagnostics and Biosensing Technologies,Medical Diagnostics & Therapeutics
5,Synthetic Biology Tools and Applications,5,0,0,Advanced Gene Regulation and Synthetic Systems,Advanced Gene Regulation & Synthetic Systems
6,Standardized Biological Parts and Engineering,6,4,0,Standardized Biological Parts and Engineering ...,Advanced Gene Regulation & Synthetic Systems
7,Probiotic Therapeutics for Diabetes,7,5,3,Synthetic Probiotics and Health Applications,Biomedical and Environmental Synthetic Biology...
8,Quorum Sensing and Biofilm Control,8,6,3,Microbial Interactions and Biofilm Control,Biomedical and Environmental Synthetic Biology...
9,Portable Environmental and Health Biosensors,9,2,2,Synthetic Diagnostics and Biosensing Technologies,Medical Diagnostics & Therapeutics


## 3. Summary

In [5]:
n_mid = hierarchy["mid_name"].notna().sum()
n_high = hierarchy["high_name"].notna().sum()
print(f"{PREFIX}: {n_mid} topics with mid_name ({hierarchy['mid_name'].dropna().nunique()} unique), "
      f"{n_high} with high_name ({hierarchy['high_name'].dropna().nunique()} unique)")

teams: 161 topics with mid_name (44 unique), 161 with high_name (4 unique)
